# DAC firmware bug checklist

Run the setup cell, then any individual case. Each case is independent and runs
against the current checked-out source. **AssertionError means the requirement
failed**. All 36 checklist cases are expected to pass after the repair phase.
No hardware is accessed by the offline tests. Physical measurements below are
separate, optional evidence; an offline pass does not establish analog behavior.

The CLI and standard-library unittest runner in `tests/README.md` are useful for repeatable
regressions. The host harness needs a C++17 compiler; it uses the actual C++ firmware
with hardware doubles. This does not substitute for compiling/flashing the Uno R4.

Run individual cells to inspect failures, or use `run_all()` to collect everything
without stopping at the first failure. Do not enable live output tests while
outputs are connected to apparatus.


In [ ]:
from pathlib import Path
import sys

# Works with a Jupyter working directory at the repository root or under tests/.
# Set ROOT explicitly if you launch the notebook from a different folder.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "Software/src/main.cpp").is_file()), None)
assert ROOT is not None, "Set ROOT to the DAC Box repository directory"
sys.path.insert(0, str(ROOT / "tests"))
from checklist_tests import run_case, run_all, PRELIMINARY, DESCRIPTIONS, MODES
print("Testing source in:", ROOT)


## Preliminary regression summary

In [ ]:
preliminary_results = run_all(PRELIMINARY, fail_on_error=True)


## Complete checklist summary
All 36 software checks should pass. Failures remain visible and are never marked expected-success.

In [ ]:
all_results = run_all()


## Mem-1 — Repeated real command dispatch performs no C++ heap allocations

Mode: **host behavior**. Counts C++ allocations across 6,000 dispatches plus source checks for allocation APIs. It does not measure real MCU heap fragmentation.

In [ ]:
run_case("Mem-1")


## Mem-2 — Silent second clients release sockets; owner survives repeated commands

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("Mem-2")


## Mem-3 — Disconnect stops socket and clears partial input before reconnect

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("Mem-3")


## Robust-1 — Reject second clients both while idle and during a ramp

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("Robust-1")


## Robust-2 — Busy handlers do no SPI work; connection lifecycle cannot release operation lock

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("Robust-2")


## C1 — Invalid channels perform no SPI; valid channels 1 through 4 work

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("C1")


## C2 — SPI begins before DAC transactions

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("C2")


## C3 — Second-client rejection leaves owner and ramp operational (overlaps Robust-1)

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("C3")


## C4 — Link/configuration loss discards stale clients and retries recovery without changing DAC outputs

Mode: **host behavior**. Injects link/configuration loss, absent hardware, and recovery. Verifies retry spacing, fresh connections, preserved DAC codes, and SAFE ZERO priority. Use the optional cable test for physical evidence.

In [ ]:
run_case("C4")


## C5 — Cooperative ramp services connections/input and aborts on disconnect (with M4/A4)

Mode: **host behavior**. Runs disconnect, command-service, pre-write abort, and second-client regressions. Long waits service input/network/buttons in bounded slices.

In [ ]:
run_case("C5")


## H1 — No local SPI settings variable shadows global settings in setup

Mode: **source audit**. Failure output states the observed discrepancy.

In [ ]:
run_case("H1")


## H2 — Reject negative ramp step before output changes

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("H2")


## H3 — Reject negative ramp delay before output changes or unsigned wait

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("H3")


## H4 — Long step waits use bounded/cooperative timing (policy check; timing needs bench)

Mode: **host behavior**. Checks bounded waits and cooperative service. B19/B20 separately verify virtual step/delay scaling; physical timing measurements below test actual accuracy.

In [ ]:
run_case("H4")


## H5 — EnableOutput public declaration matches definition

Mode: **source audit**. Failure output states the observed discrepancy.

In [ ]:
run_case("H5")


## H6 — Bare plus/minus signs are not integers

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("H6")


## M1 — Expire idle clients after 120 seconds while preserving active ramps

Mode: **host behavior**. The implemented policy releases an idle owner after 120 seconds without received bytes, while keeping long active ramps alive. Covers renewal, reconnect, and timer rollover.

In [ ]:
run_case("M1")


## M2 — Client receives explicit warning/error when target is clamped

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("M2")


## M3 — Oversized command cannot execute a truncated valid prefix

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("M3")


## M4 — Command injected during ramp is serviced before ramp returns

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("M4")


## M5 — Local include capitalization matches actual filename

Mode: **source audit**. Failure output states the observed discrepancy.

In [ ]:
run_case("M5")


## M6 — No unused frame-prefix assignments

Mode: **source audit**. Failure output states the observed discrepancy.

In [ ]:
run_case("M6")


## M7 — GPIO aliases resolve to defined symbols

Mode: **source audit**. Failure output states the observed discrepancy.

In [ ]:
run_case("M7")


## M8 — Balanced foreground SPI transactions and SPI-free cancellation hooks

Mode: **source + host ownership audit**. Reviewed as no current ISR contention defect: verifies balanced foreground SPI transactions, absence of application SPI ISRs, and SPI-free cancellation hooks. A future SPI ISR needs a new ownership review.

In [ ]:
run_case("M8")


## M9 — INIT preserves active outputs and verifies cold initialization at zero

Mode: **host behavior**. INIT preserves normal active channels without writes. Cold ground-clamped channels are configured and verified at zero before release; incompatible active modes fail unchanged. Scope capture remains supplemental.

In [ ]:
run_case("M9")


## M10 — Reject integer overflow before DAC access

Mode: **host behavior**. Huge integers are tested on the host only. Strict signed-32-bit parsing must reject overflow before any DAC access.

In [ ]:
run_case("M10")


## M11 — Acknowledge ramp before first blocking delay

Mode: **host behavior**. Failure output states the observed discrepancy.

In [ ]:
run_case("M11")


## A1 — Active client is not reclassified as a second client

Mode: **host behavior**. Codex addition: existing connection reclassified as a second client. This overlaps Mem-2 and is repaired in that preliminary fix.

In [ ]:
run_case("A1")


## A2 — INIT detects bad register readback instead of reporting success

Mode: **host behavior**. Codex addition: injects corrupt readback and ignored register writes; INIT must report failure and retain a cold channel's clamp.

In [ ]:
run_case("A2")


## A3 — Repository contains a reproducible target/dependency build configuration

Mode: **source audit**. Codex addition: audits exact Uno R4/platform/core/compiler/uploader versions and the vendored Ethernet fork. A separate clean target build is recorded in VALIDATION.md.

In [ ]:
run_case("A3")


## A4 — No additional DAC write after disconnect during step delay

Mode: **host behavior**. Codex addition: disconnect during the simulated step delay must prevent the next DAC write.

In [ ]:
run_case("A4")


## A5 — Ethernet waits are bounded and cancellable; recovery clears controller/socket state

Mode: **host behavior**. Codex addition: bounded/cancellable actual Ethernet-driver waits, chip/PHY reset, socket-cache cleanup, and a stopped-timebase fallback.

In [ ]:
run_case("A5")


## A6 — USB diagnostic backpressure does not block physical controls

Mode: **host behavior**. Codex addition: USB diagnostic backpressure must not prevent commands or local button servicing.

In [ ]:
run_case("A6")


## A7 — Uno R4 startup leaves SD deselected on the shared SPI bus

Mode: **host behavior**. Codex-introduced regression, fixed: models Uno R4 OUTPUT mode clearing the latch; SD must be deselected before SPI traffic.

In [ ]:
run_case("A7")


## A8 — Malformed readback cannot become a voltage or a ramp starting value

Mode: **host behavior**. Codex addition: invalid register frames must fail READ and both ramp paths without output writes. Valid full-scale codes remain accepted.

In [ ]:
run_case("A8")


## A9 — Python V/s conversion and acceptance/status protocol are validated offline

Mode: **offline Python behavior**. Client-wrapper addition: checks V/s-to-microseconds conversion, voltage boundaries, input rejection, acceptance/status polling, aborts, and timeouts without a connection.

In [ ]:
run_case("A9")


## Optional live bench setup — disabled by default

Set the address yourself. These cells do not upload firmware. Record the revision
flashed on your board; otherwise the test does not establish which source it ran.
Only set `outputs_isolated` and `allow_output_changes` after disconnecting the
test output from apparatus. Choose a small explicit code window; no default
voltage is assumed. Output tests leave the last code in place and do not call INIT.
Malformed ramp/overflow and driver fault-injection cases stay in the offline suite.


In [ ]:
from live_bench import (Bench, verify_step_intervals, verify_init_waveform,
                        verify_bus_capture)
bench = Bench(
    host="",                       # e.g. the configured static board IP
    enabled=False,                 # opt in to real TCP connections
    outputs_isolated=False,
    allow_output_changes=False,    # separate opt-in to output changes
    channel=1,
    low_code=None,                 # choose approved codes, not voltages
    high_code=None,                # span 100..1000 for the ramp checks
    timeout=3.0,
)
flashed_revision = ""              # record manually; ID does not expose firmware version
print("Live tests enabled:", bench.enabled)


### Read-only continuity (Mem-1 soak supplement, C2, A1)

In [ ]:
if bench.enabled:
    bench.readonly(repeats=1000)
else:
    print("NOT RUN: live test disabled")


### Silent second-client/socket stress (Mem-2, Robust-1, C3)

In [ ]:
if bench.enabled:
    bench.reject_extra_clients()
else:
    print("NOT RUN: live test disabled")


### Reconnect after partial command (Mem-3)

In [ ]:
if bench.enabled:
    bench.reconnect_partial_command()
else:
    print("NOT RUN: live test disabled")


### Invalid READ channels (C1)

In [ ]:
if bench.enabled:
    bench.invalid_channels()
else:
    print("NOT RUN: live test disabled")


### Output register SET/READ
Changes output and leaves its final code in place.

In [ ]:
if bench.enabled and bench.allow_output_changes:
    bench.set_readback()
else:
    print("NOT RUN: live test disabled")


### Second-client rejection during ramp (Robust-1, C3)
Changes output and leaves its final code in place.

In [ ]:
if bench.enabled and bench.allow_output_changes:
    bench.ramp_and_reject()
else:
    print("NOT RUN: live test disabled")


### Ramp disconnect and lock release (C5, Robust-2)
Changes output and leaves its final code in place.

In [ ]:
if bench.enabled and bench.allow_output_changes:
    bench.abort_ramp()
else:
    print("NOT RUN: live test disabled")


### Manual cable recovery (C4)
Enable this separately: it prompts you to unplug and reconnect the cable, then verifies recovery using a fresh TCP connection.

In [ ]:
RUN_CABLE_TEST = False
if bench.enabled and RUN_CABLE_TEST:
    bench.cable_recovery()
else:
    print("NOT RUN: manual cable test disabled")


## External measurement assertions

These are **not simulated measurements**. Fill in values from a scope/logic analyzer
and retain the original capture with firmware revision, instrument settings,
sample rate, and chosen acceptance limits. Empty data is reported as NOT RUN.
For M8, capture shared-bus activity during network and DAC work; include any SPI
interrupt workload added in future firmware. The M9 trace must
cover before INIT through settling; choose capture resolution suitable for glitches.


In [ ]:
# H4: measured time between actual DAC updates, in microseconds.
step_intervals_us = []
requested_step_us = None
timing_tolerance_us = None
if step_intervals_us:
    verify_step_intervals(step_intervals_us, requested_step_us, timing_tolerance_us)
else:
    print("NOT RUN H4: supply measured step intervals and chosen tolerance")


In [ ]:
# M9: scope samples spanning INIT, with application-approved bounds.
init_waveform_volts = []
approved_min_volts = None
approved_max_volts = None
if init_waveform_volts:
    verify_init_waveform(init_waveform_volts, approved_min_volts, approved_max_volts)
else:
    print("NOT RUN M9: supply measured INIT waveform and approved limits")


In [ ]:
# M8: analyzed shared-SPI capture under the relevant interrupt workload.
bus_capture = None  # e.g. dict(overlapping_cs_frames=0, malformed_frames=0, tested_frames=1000)
if bus_capture is not None:
    verify_bus_capture(**bus_capture)
else:
    print("NOT RUN M8: supply analyzed bus-capture results")
